# Opdracht 3 · Logistische regressie & interpretatie
### Regressie-training · Cursist-versie

Logistische regressie voorspelt een **categorie** in plaats van een getal. In deze opdracht ga je:

1. Een **logistisch regressiemodel** trainen op een classificatieprobleem
2. De **confusion matrix** maken op de test-set
3. **Accuracy, precision en recall** berekenen en het verschil interpreteren
4. De **ROC-curve** tekenen en de **AUC** aflezen
5. Eén coëfficiënt omzetten naar een **odds ratio** en uitleggen

> **Dataset:** de Breast Cancer-dataset uit scikit-learn (geen download nodig). Op basis van metingen
> van een tumor voorspellen we of die **kwaadaardig** (positief) of **goedaardig** is.
> We kiezen *kwaadaardig* als de positieve klasse, want dat is het geval dat je vooral niet wilt missen.

> **Hoe werkt dit notebook?** De meeste cellen zijn ingevuld; op de plekken met **`# TODO`** vul jij iets in.


## 0. Voorbereiding
Imports en data laden. Gewoon uitvoeren.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (confusion_matrix, ConfusionMatrixDisplay,
                             accuracy_score, precision_score, recall_score,
                             roc_curve, roc_auc_score)

plt.rcParams["figure.figsize"] = (6, 4)
print("Bibliotheken geladen.")

In [ ]:
# Data laden. We zetten de positieve klasse op 'kwaadaardig'.
data = load_breast_cancer(as_frame=True)
df = data.frame

# In de originele data is target=1 'goedaardig' en target=0 'kwaadaardig'.
# We draaien dit om, zodat y=1 betekent: KWAADAARDIG (de positieve klasse).
y = 1 - df["target"]
X = df.drop(columns=["target"])

print("Aantal rijen:", X.shape[0], "| aantal features:", X.shape[1])
print("Kwaadaardig (y=1):", int(y.sum()), " | goedaardig (y=0):", int((y == 0).sum()))

We splitsen in train/test. Met `stratify=y` houden we de verhouding kwaadaardig/goedaardig
in beide sets gelijk — belangrijk bij classificatie.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Logistische regressie werkt beter met geschaalde features.
scaler = StandardScaler().fit(X_train)
X_train_s = scaler.transform(X_train)
X_test_s = scaler.transform(X_test)
print("Train:", X_train.shape[0], "| Test:", X_test.shape[0])

## 1. Een logistisch model trainen

### TODO 1 — Train het model
Maak een `LogisticRegression`-model (gebruik `max_iter=5000` zodat het zeker convergeert),
train het op de **geschaalde** train-set, en voorspel de **geschaalde** test-set.

> Tip: `LogisticRegression(max_iter=5000)`, dan `.fit(X_train_s, y_train)` en `.predict(X_test_s)`.

In [ ]:
# TODO 1: train een logistisch model en voorspel de test-set
model = ...
model.fit(...)
y_pred = model.predict(...)

# Voorspelde kansen op de positieve klasse (al ingevuld - nodig voor de ROC-curve)
y_proba = model.predict_proba(X_test_s)[:, 1]
print("Model getraind. Eerste 5 voorspellingen:", y_pred[:5])

## 2. Confusion matrix
De confusion matrix zet werkelijke tegen voorspelde klassen. Hieruit volgen alle andere metrieken:
- **TP** (true positive): kwaadaardig, ook zo voorspeld
- **FN** (false negative): kwaadaardig, maar gemist — klinisch het gevaarlijkst
- **FP** (false positive): goedaardig, maar als kwaadaardig bestempeld
- **TN** (true negative): goedaardig, ook zo voorspeld

### TODO 2 — Maak de confusion matrix
Bereken de confusion matrix door `y_test` met `y_pred` te vergelijken.

> Tip: `confusion_matrix(y_test, y_pred)`.

In [ ]:
# TODO 2: bereken de confusion matrix
cm = confusion_matrix(..., ...)
print(cm)

ConfusionMatrixDisplay(cm, display_labels=["goedaardig", "kwaadaardig"]).plot(cmap="Blues")
plt.title("Confusion matrix")
plt.show()

## 3. Accuracy, precision en recall
- **Accuracy**: aandeel correcte voorspellingen.
- **Precision**: van alles wat we *kwaadaardig* noemden, hoeveel klopte er? (weinig vals alarm)
- **Recall**: van alle werkelijk *kwaadaardige* gevallen, hoeveel vingen we er? (weinig missers)

In een medische context weegt **recall** vaak zwaar: een gemiste tumor (false negative) is ernstiger
dan een vals alarm.

### TODO 3 — Bereken de drie metrieken
Gebruik `accuracy_score`, `precision_score` en `recall_score`, telkens met `y_test` en `y_pred`.

In [ ]:
# TODO 3: bereken accuracy, precision en recall
acc = accuracy_score(..., ...)
prec = precision_score(..., ...)
rec = recall_score(..., ...)

print(f"Accuracy : {acc:.3f}")
print(f"Precision: {prec:.3f}")
print(f"Recall   : {rec:.3f}")

> **Interpreteer:** is je recall hoger of lager dan je precision? Kijk terug naar de confusion matrix:
> hoeveel kwaadaardige gevallen heeft het model gemist (false negatives)?

## 4. ROC-curve & AUC
De ROC-curve toont de afruil tussen het vinden van positieve gevallen (true-positive-rate) en vals alarm
(false-positive-rate), over alle mogelijke drempels. De **AUC** vat dit samen in één getal:
0,5 = gokken, 1,0 = perfect.

### TODO 4 — Bereken de AUC
Bereken de AUC met `roc_auc_score`, op basis van `y_test` en de voorspelde kansen `y_proba`.

> Let op: gebruik hier de **kansen** (`y_proba`), niet de harde voorspellingen (`y_pred`).

In [ ]:
# TODO 4: bereken de AUC met de voorspelde kansen
auc = roc_auc_score(..., ...)
print(f"AUC: {auc:.3f}")

De ROC-curve zelf is al voor je getekend — voer de cel uit.

In [ ]:
# ROC-curve (al ingevuld)
fpr, tpr, drempels = roc_curve(y_test, y_proba)
plt.plot(fpr, tpr, color="#4473C5", linewidth=2.5, label=f"model (AUC = {auc:.3f})")
plt.plot([0, 1], [0, 1], color="#9199B9", linestyle="--", label="gokken (AUC = 0,5)")
plt.xlabel("False-positive-rate")
plt.ylabel("True-positive-rate")
plt.title("ROC-curve")
plt.legend(loc="lower right")
plt.show()

## 5. Van coëfficiënt naar odds ratio
Een coëfficiënt in logistische regressie zit op de **log-odds-schaal**. Door er `e^β` van te nemen,
krijg je een **odds ratio**: het effect op de *odds* van de positieve klasse per eenheid toename
van die feature (de features zijn hier geschaald).

- Odds ratio > 1 → feature verhoogt de odds op kwaadaardig
- Odds ratio < 1 → feature verlaagt de odds
- Odds ratio = 1 → geen effect

### TODO 5 — Bereken de odds ratio van de sterkste feature
1. De cel hieronder vindt al de feature met de grootste absolute coëfficiënt.
2. Bereken de odds ratio: `np.exp(...)` van die coëfficiënt.

> Tip: `np.exp(sterkste_coef)`.

In [ ]:
# Sterkste feature opzoeken (al ingevuld)
coefs = pd.Series(model.coef_[0], index=X.columns)
sterkste_feature = coefs.abs().idxmax()
sterkste_coef = coefs[sterkste_feature]
print(f"Sterkste feature: '{sterkste_feature}'  (coëfficiënt = {sterkste_coef:.3f})")

In [ ]:
# TODO 5: zet de coëfficiënt om naar een odds ratio
odds_ratio = np.exp(...)
print(f"Odds ratio: {odds_ratio:.3f}")

> **Interpreteer:** is de odds ratio groter of kleiner dan 1? Wat betekent dat voor het effect van
> deze feature op de kans dat een tumor kwaadaardig is?

## 6. Bespreking
Bespreek met je buur:

- Welke metriek vind je hier het belangrijkst — accuracy, precision of recall? Waarom?
- Hoeveel kwaadaardige gevallen miste het model? Wat zou je doen om die missers te verminderen?
- Wat zegt de AUC over de kwaliteit van het model?
- Als je de **beslissingsdrempel** zou verlagen (sneller 'kwaadaardig' zeggen), wat gebeurt er dan
  met recall en precision?
- Wat betekent de odds ratio die je berekende, in gewone taal?